In [4]:
import { createAgent, tool } from "npm:langchain";

import { ChatGoogle } from "npm:@langchain/google";

import { z } from "npm:zod";

import { GoogleGenerativeAIEmbeddings } from "npm:@langchain/google-genai";

import { MemoryVectorStore } from "npm:@langchain/classic/vectorstores/memory";

import { Document } from "npm:@langchain/core/documents";

import { parse } from "jsr:@std/dotenv";


In [16]:
const env = parse(await Deno.readTextFile(".env"));

const GOOGLE_API_KEY = env.GOOGLE_API_KEY;
const API_URL = env.RESTAURANT_API_URL;

const model = new ChatGoogle({
  model: "gemini-3.1-flash-lite",
  apiKey: GOOGLE_API_KEY,
  temperature: 0,
});

type RestaurantCategory =
  | "한식"
  | "일식"
  | "중식"
  | "양식"
  | "세계요리"
  | "특별한 술집"
  | "전통차/커피전문점"
  | "디저트/베이커리";

type Restaurant = {
  OPENDATA_ID: string;
  BZ_NM: string;
  GNG_CS: string;
  FD_CS: string;
  TLNO: string;
  MBZ_HR: string;
  SEAT_CNT: string;
  PKPL: string;
  HP: string;
  PSB_FRN: string;
  BKN_YN: string;
  INFN_FCL: string;
  BRFT_YN: string;
  DSSRT_YN: string;
  MNU: string;
  SMPL_DESC: string;
  SBW: string;
  BUS: string;
};

// 외부api 호출 함수
async function fetchRestaurants(district: string): Promise<Restaurant[]> {
  const url = new URL(API_URL);

  url.searchParams.set("addr", district);

  const response = await fetch(url);

  if (!response.ok) {
    throw new Error(`API 요청 실패: ${response.status}`);
  }

  const data = await response.json();

  // 실제 API 응답 구조에 맞게 수정
  return data.data as Restaurant[];
}

// 카테고리 필터 함수
function filterByCategory(
  restaurants: Restaurant[],
  category: RestaurantCategory | null,
): Restaurant[] {
  if (category === null) {
    return restaurants;
  }

  return restaurants.filter((restaurant) => restaurant.FD_CS === category);
}

//필터링된 가게만 출력하는함수
async function searchRestaurants(
  district: string,
  category: RestaurantCategory | null,
) {
  const restaurants = await fetchRestaurants(district);

  const filtered = filterByCategory(restaurants, category);

  return filtered;
}

const result = await searchRestaurants("남구", "한식");

console.log(result);

const embeddings = new GoogleGenerativeAIEmbeddings({
  model: "gemini-embedding-001",
  apiKey: GOOGLE_API_KEY,
});

function createRestaurantDocument(restaurant: Restaurant) {
  const content = `
${restaurant.BZ_NM}은(는)
${restaurant.GNG_CS}에 위치한
${restaurant.FD_CS} 음식점입니다.

${restaurant.SMPL_DESC}

좌석 정보는 ${restaurant.SEAT_CNT}입니다.
주차 정보는 ${restaurant.PKPL}입니다.
예약은 ${restaurant.BKN_YN}합니다.

${restaurant.SBW}
${restaurant.BUS}
  `.trim();

  return new Document({
    pageContent: content,

    metadata: {
      id: restaurant.OPENDATA_ID,

      name: restaurant.BZ_NM,

      category: restaurant.FD_CS,

      address: restaurant.GNG_CS,
    },
  });
}

const restaurants = await fetchRestaurants("중구");
const categoryFiltered = filterByCategory(restaurants, "한식");

const documents = categoryFiltered.map(createRestaurantDocument);

// 벡터 스토어 저장
const vectorStore = await MemoryVectorStore.fromDocuments(
  documents,
  embeddings,
);



[
  {
    cnt: "1",
    OPENDATA_ID: "1779",
    GNG_CS: "대구광역시 남구 대명동 1649-4",
    FD_CS: "한식",
    BZ_NM: "청라연",
    TLNO: "053-629-0011",
    MBZ_HR: "11:00 ~ 21:20(15:00~17:00 브레이크타임)",
    SEAT_CNT: "150석(룸8)",
    PKPL: "24대",
    HP: "없음",
    PSB_FRN: "영어 ",
    BKN_YN: "가능",
    INFN_FCL: "불가능",
    BRFT_YN: "불가능",
    DSSRT_YN: "가능",
    MNU: "다올상 22,000원 <br />해가빛상 34,000원 <br />모꼬지상 45,000원 <br />약선갈비찜 38,000원 <br />약선장어구이 33,000원 <br />약선모둠수육 33,000원<br />",
    SMPL_DESC: "청라연 은 한정식 전문점으로 자연 그대로의 맛을 살리기 위해 노력하는 한식당입니다.",
    SBW: "지하철 1호선 안지랑역 1번 출구에서 도보로 약 564m 거리.",
    BUS: "버스 정류장은 대명1동행정복지센터 정류장이 가장 가깝습니다."
  },
  {
    cnt: "3",
    OPENDATA_ID: "1721",
    GNG_CS: "대구광역시 남구 대명동 1594-15",
    FD_CS: "한식",
    BZ_NM: "바다와대림",
    TLNO: "053-629-3387",
    MBZ_HR: "12:00 ~ 24:00",
    SEAT_CNT: "44석(룸2)",
    PKPL: "없음",
    HP: "없음",
    PSB_FRN: "가능한 외국어가 없습니다.",
    BKN_YN: "가능",
    INFN_FCL: "불가능",
    BRFT_YN: "불가능",
    DSSRT_YN: "가능",
    MNU: "산낙지 철판 볶음(1인분) 

In [17]:
const restaurantSearchTool = tool(
  // 실제로 실행할 함수
  async ({ district, category }) => {
    const restaurants = await searchRestaurants(district, category);

    return {
      count: restaurants.length,

      restaurants: restaurants.map((restaurant) => ({
        name: restaurant.BZ_NM,

        category: restaurant.FD_CS,

        address: restaurant.GNG_CS,

        menu: restaurant.MNU,

        parking: restaurant.PKPL,

        reservation: restaurant.BKN_YN,

        seats: restaurant.SEAT_CNT,

        description: restaurant.SMPL_DESC,
      })),
    };
  },
  // tool의 설명
  {
    name: "search_restaurants",

    description:
      "대구광역시의 특정 지역에서 음식점, 카페, 술집 등을 공공데이터 API로 검색합니다.",

    schema: z.object({
      district: z
        .string()
        .describe("검색할 대구광역시 행정구역. 예: 남구, 중구, 수성구"),

      category: z
        .enum([
          "한식",
          "일식",
          "중식",
          "양식",
          "세계요리",
          "특별한 술집",
          "전통차/커피전문점",
        ])
        .nullable()
        .describe("검색할 식당 카테고리. 특정 카테고리가 없으면 null"),
    }),
  },
);


In [19]:
const budgetCheckTool = tool(
  async ({ restaurants, budgetMin, budgetMax }) => {
    const results = restaurants.map((restaurant) => {
      const matches = restaurant.menu.matchAll(
        /(\d{1,3}(?:,\d{3})+|\d+)\s*원/g,
      );

      const prices = Array.from(matches).map((match) =>
        Number(match[1].replaceAll(",", "")),
      );

      const matchedPrices = prices.filter(
        (price) => price >= budgetMin && price <= budgetMax,
      );

      return {
        name: restaurant.name,

        matched: matchedPrices.length > 0,

        matchedPrices,
      };
    });

    return {
      budgetMin,
      budgetMax,

      restaurants: results.filter((restaurant) => restaurant.matched),
    };
  },

  {
    name: "check_budget",

    description: `
search_restaurants Tool로 검색한 식당들의
메뉴 가격이 사용자의 1인당 예산 범위에
맞는지 확인합니다.

사용자가 가격이나 예산 조건을 말한 경우에만 사용하세요.

이 Tool을 사용하기 전에
먼저 search_restaurants를 사용하여
실제 식당 정보를 확보해야 합니다.
`,

    schema: z.object({
      restaurants: z
        .array(
          z.object({
            name: z.string(),

            menu: z.string(),
          }),
        )
        .describe("search_restaurants가 반환한 식당 이름과 메뉴 정보"),

      budgetMin: z.number().describe("사용자의 1인당 최소 예산"),

      budgetMax: z.number().describe("사용자의 1인당 최대 예산"),
    }),
  },
);


RAG를 tool로 변환하여 적용하기

In [20]:
const preferenceSearchTool = tool(
  async ({ query, candidateNames }) => {
    const candidateSet = new Set(candidateNames);

    const filter = (document: any) => {
      return candidateSet.has(document.metadata.name);
    };
    // 전체 벡터스토어에서 후보 식당만 유사도를 검사
    const results = await vectorStore.similaritySearch(query, 3, filter);

    return {
      query,

      count: results.length,

      restaurants: results.map((document) => ({
        name: document.metadata.name,

        category: document.metadata.category,

        address: document.metadata.address,

        description: document.pageContent,
      })),
    };
  },

  {
    name: "search_preferences",

    description: `
사용자의 취향이나 분위기 조건에 맞는
식당을 의미 검색하는 RAG Tool입니다.

조용한 곳,
부모님과 가기 좋은 곳,
데이트하기 좋은 곳,
가족 모임에 좋은 곳 등의 조건이 있으면
반드시 사용해야 합니다.

정확한 지역이나 가격 검색에는 사용하지 않습니다.
`,

    schema: z.object({
      query: z
        .string()
        .describe(
          "검색할 취향이나 분위기 조건. 예: 부모님과 조용하게 식사할 곳",
        ),

      candidateNames: z
        .array(z.string())
        .describe("앞선 Tool에서 검색된 후보 음식점 이름 목록"),
    }),
  },
);


In [9]:
const ragResult = await preferenceSearchTool.invoke({
  query: "부모님 모시고 조용하게 식사할 곳",
  candidateNames: ["청라연", "OO한정식", "ABC식당"],
});

console.log(ragResult);


{ query: "부모님 모시고 조용하게 식사할 곳", count: 0, restaurants: [] }


지금까지 만든 도구 등록

In [21]:
const agent = createAgent({
  model,

  tools: [restaurantSearchTool, budgetCheckTool, preferenceSearchTool],

  systemPrompt:  `
당신은 대구 맛집 추천 AI 에이전트입니다.

반드시 다음 규칙을 따르세요.

[1. 식당 검색]

사용자가 실제 식당을 찾거나 추천을 요청하면
반드시 search_restaurants를 먼저 사용합니다.


[2. 예산 조건]

사용자의 질문에
가격, 예산, 금액 조건이 포함되어 있다면
search_restaurants 결과를 받은 후
반드시 check_budget을 사용합니다.

메뉴 문자열을 직접 읽고
LLM이 가격을 판단하지 마세요.
가격 판정은 반드시 check_budget Tool에게 맡기세요.


[3. 취향 및 분위기 조건]

사용자의 질문에 다음과 같은
취향이나 분위기 조건이 포함되어 있다면

- 조용한
- 부모님과 방문
- 데이트
- 가족 모임
- 분위기 좋은
- 편안한

반드시 search_preferences를 사용합니다.

식당 description을 LLM이 직접 읽고
취향 적합성을 최종 판단하지 마세요.

취향 검색은 반드시
search_preferences Tool에게 맡기세요.


[4. Tool 실행 순서]

예산과 취향 조건이 모두 있다면:

search_restaurants
→ check_budget
→ search_preferences

순서로 실행합니다.

앞 단계의 결과를
다음 Tool의 후보로 전달하세요.


[5. 최종 답변]

필요한 Tool 실행이 모두 끝난 뒤에만
최종 식당 추천 답변을 생성하세요.

Tool에서 확인되지 않은 정보는
만들어내지 마세요.
`,
});


In [25]:
const test1 =  "대구 남구에서 한식집 추천해줘";
const test2 = "대구 남구에서 부모님 모시고 갈 조용한 한식집 추천해줘";
const test3 = `
대구 남구에서
1인 3만원 정도로 먹을 수 있고
부모님 모시고 갈
조용한 한식집 추천해줘
`;

const result = await agent.invoke({
  messages: [
    {
      role: "user",

      content: test3
    },
  ],
});


In [26]:
//호출된 툴 확인

for (const message of result.messages) {
  if (
    "tool_calls" in message &&
    Array.isArray(message.tool_calls) &&
    message.tool_calls.length > 0
  ) {
    console.log("Tool Calls:", message.tool_calls);
  }
}


Tool Calls: [
  {
    type: "tool_call",
    id: "call_481355",
    name: "search_restaurants",
    args: { category: "한식", district: "남구" },
    thoughtSignature: "EnEKbwERTTIPhL6pr7jfNx3auvVenFWJ+tk2iXRFmBlDz86JHzaRjfUvI3tZHLl7qd8IJTurM3MpciqkUppK7dpkAjSsvBZe15lLvS+fgYqGECY4KCH63u6TOutvTISJGqXfTsg6HI10aiuiD//BDUoCpg=="
  }
]
Tool Calls: [
  {
    type: "tool_call",
    id: "call_469581",
    name: "check_budget",
    args: {
      budgetMin: 20000,
      budgetMax: 30000,
      restaurants: [
        {
          name: "청라연",
          menu: "다올상 22,000원 <br />해가빛상 34,000원 <br />모꼬지상 45,000원 <br />약선갈비찜 38,000원 <br />약선장어구이 33,000원 <br />약선모둠수육 33,000원<br />"
        },
        {
          menu: "소갈비찜정식 20,000원 <br />소갈비찜+돌솥밥 22,000원 <br />능이돌솥밥정식10,000원 <br />뚝배기불고기정식11,000원 <br />차돌청국장정식 8,000원 <br />차돌된장정식 7,000원 <br />누룽지탕 7,000원<br />",
          name: "일미정"
        },
        {
          menu: "행복한정식+돌솥밥 14,000원 <br />돼지주물럭+돌솥밥 16,000원 <br />김치찜+돌솥밥 15,000원 <br />김치찌개+돌솥밥 15,000

In [27]:
const lastMessage = result.messages.at(-1);

console.log(lastMessage?.content);


대구 남구에서 부모님을 모시고 조용하게 식사하기 좋은 한식당을 추천해 드립니다. 요청하신 1인당 3만 원 내외의 예산으로 이용 가능한 곳들입니다.

*   **청라연**: 한정식 전문점으로, 자연의 맛을 살린 정갈한 상차림을 제공합니다. 룸(8개)이 마련되어 있어 부모님과 오붓하고 조용하게 식사하기 좋습니다. (다올상 22,000원 등)
*   **일미정**: 약선 음식 전문점으로, 몸에 좋은 약선 양념을 사용한 요리를 맛볼 수 있습니다. 룸이 있어 가족 모임에 적합합니다. (소갈비찜 정식 20,000원~22,000원 등)
*   **물베기한정식**: 다양한 정식 메뉴를 갖추고 있으며, 룸(6개)이 있어 조용한 분위기에서 식사하기 좋습니다. (들국화정식 22,000원, 수선화정식 27,000원 등)
*   **백복수반**: 아늑한 황토방 분위기와 전통적인 소품들로 꾸며져 있어 편안하고 정감 있는 식사가 가능합니다. (불고기정식 20,000원 등)

방문하시기 전에 미리 예약하시면 더욱 편안하게 이용하실 수 있습니다. 즐거운 식사 시간 되시길 바랍니다.
